# 04 Ising formulation and annealing scaffold

Этот ноутбук переводит QUBO-постановку universal hall-aware сценария в Ising-форму и готовит вход для quantum-inspired / simulated annealing.

На этом этапе мы:
- загружаем сохранённые артефакты QUBO (mapping переменных, матрицу Q, candidate_df),
- переводим QUBO-энергию E(x) в Ising-энергию E(s) через стандартное преобразование x_i = (1 + s_i) / 2,
- проверяем, что энергии QUBO и Ising отличаются только на константу для нескольких тестовых конфигураций,
- подготавливаем функции ising_energy и decode_spins_to_schedule, которые будут использоваться annealer'ом.

## Навигация по ноутбуку

### Зачем нужен этот ноутбук
Этот ноутбук переводит сохранённую QUBO-постановку в Ising-форму и запускает annealing-based solvers: simulated annealing, OpenJij SA и OpenJij SQA. Это мост между classical baseline и будущими QAOA-экспериментами.

### Что здесь самое важное
- Проверить, что энергии QUBO и Ising совпадают с точностью до константы.
- Использовать ту же самую задачу, что и в notebook 03, чтобы сравнение solver'ов было честным.
- Оценивать не только энергию, но и качество декодированного расписания: revenue, число сеансов и число конфликтов.

### Какие результаты здесь ключевые
- Коэффициенты Ising `(h, J)`, полученные из QUBO.
- Результаты annealing-решателей после декодирования обратно в расписание.
- Сравнение с classical baseline по бизнес-метрикам и по соблюдению ограничений.

### Как читать этот ноутбук
Этот ноутбук носит исследовательский характер, но трактовать результаты нужно строго. Хорошим считается не просто решение с низкой энергией, а решение, которое после декодирования даёт сильное расписание без физических конфликтов.


In [78]:
from __future__ import annotations

from pathlib import Path
import json
from typing import Dict, Tuple, List

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
QUBO_DIR = PROJECT_ROOT / 'data' / 'qubo'

QUBO_DIR, PROJECT_ROOT


(PosixPath('/Users/mishatrubik/Desktop/QC/BOQ/data/qubo'),
 PosixPath('/Users/mishatrubik/Desktop/QC/BOQ'))

In [79]:
# Загрузка QUBO артефактов
with (QUBO_DIR / 'universal_qubo_var_index_map.json').open('r', encoding='utf-8') as f:
    qubo_var_index_map: Dict[str, int] = json.load(f)

with (QUBO_DIR / 'universal_qubo_matrix.json').open('r', encoding='utf-8') as f:
    qubo_Q_raw = json.load(f)

universal_candidate_scored_df = pd.read_csv(
    QUBO_DIR / 'universal_candidate_scored.csv',
    parse_dates=['show_datetime', 'end_datetime'],
)

len(qubo_var_index_map), len(qubo_Q_raw), universal_candidate_scored_df.shape


(32, 166, (32, 30))

In [80]:
# Преобразуем qubo_Q_raw в удобную структуру и определим число переменных
num_variables = len(qubo_var_index_map)
# Приведём ключи mapping к int, если они были сериализованы как строки
qubo_var_index_map_int: Dict[int, int] = {int(k): int(v) for k, v in qubo_var_index_map.items()}

# QUBO-матрица в формате списка троек (i, j, q_ij)
qubo_Q: List[Tuple[int, int, float]] = [
    (int(term['i']), int(term['j']), float(term['q'])) for term in qubo_Q_raw
]

qubo_Q

[(0, 0, -23940.239999999998),
 (1, 1, -26463.429999999997),
 (2, 2, -24030.239999999998),
 (3, 3, -23999.04),
 (4, 4, -23940.239999999998),
 (5, 5, -26463.429999999997),
 (6, 6, -24030.239999999998),
 (7, 7, -23999.04),
 (8, 8, -40897.909999999996),
 (9, 9, -41442.729999999996),
 (10, 10, -41051.659999999996),
 (11, 11, -40998.36),
 (12, 12, -40897.909999999996),
 (13, 13, -41442.729999999996),
 (14, 14, -41051.659999999996),
 (15, 15, -40998.36),
 (16, 16, -46882.97),
 (17, 17, -51428.92999999999),
 (18, 18, -47059.22),
 (19, 19, -46498.14),
 (20, 20, -46882.97),
 (21, 21, -51428.92999999999),
 (22, 22, -47059.22),
 (23, 23, -46498.14),
 (24, 24, -80798.31),
 (25, 25, -81886.84),
 (26, 26, -82103.31999999999),
 (27, 27, -81996.72),
 (28, 28, -80798.31),
 (29, 29, -81886.84),
 (30, 30, -82103.31999999999),
 (31, 31, -81496.74),
 (0, 1, 400000.0),
 (0, 2, 400000.0),
 (0, 3, 400000.0),
 (0, 4, 400000.0),
 (0, 5, 400000.0),
 (0, 6, 400000.0),
 (0, 7, 400000.0),
 (1, 2, 400000.0),
 (1, 3, 

## QUBO → Ising преобразование

Мы используем стандартную связь между бинарными переменными x_i ∈ {0,1} и спинами s_i ∈ {-1, 1}:

x_i = (1 + s_i) / 2.

Подставляя эту связь в QUBO-энергию E(x) = ∑_i Q_ii x_i + ∑_{i<j} Q_ij x_i x_j,
получаем Ising-энергию E(s) = const + ∑_i h_i s_i + ∑_{i<j} J_ij s_i s_j.

In [81]:
def qubo_to_ising(
    qubo_terms: List[Tuple[int, int, float]],
    num_vars: int,
) -> Tuple[np.ndarray, np.ndarray, float]:
    """
    Преобразует QUBO-представление (список (i, j, q_ij)) в Ising-форму:
    E_QUBO(x) = ∑_{i≤j} Q_ij x_i x_j
    E_Ising(s) = const + ∑_i h_i s_i + ∑_{i<j} J_ij s_i s_j,
    где x_i = (1 + s_i) / 2.
    """
    # Инициализируем h и J нулями
    h = np.zeros(num_vars, dtype=float)
    J = np.zeros((num_vars, num_vars), dtype=float)
    const = 0.0

    for i, j, q in qubo_terms:
        if i == j:
            # Диагональный элемент: q * x_i
            # x_i = (1 + s_i)/2 => q * x_i = q/2 + (q/2) * s_i
            const += q / 2.0
            h[i] += q / 2.0
        else:
            # Вне диагонали: q * x_i * x_j
            # x_i x_j = ((1 + s_i)/2) * ((1 + s_j)/2)
            #          = 1/4 * (1 + s_i + s_j + s_i s_j)
            # => вклад в энергию: q/4 * 1 + q/4 * s_i + q/4 * s_j + q/4 * s_i s_j
            const += q / 4.0
            h[i] += q / 4.0
            h[j] += q / 4.0
            J[i, j] += q / 4.0
            J[j, i] += q / 4.0

    return h, J, const


h_vec, J_mat, qubo_const = qubo_to_ising(qubo_Q, num_variables)

{
    'num_variables': num_variables,
    'nonzero_J': int(np.count_nonzero(J_mat) / 2),
}


{'num_variables': 32, 'nonzero_J': 134}

In [82]:
def qubo_energy(x: np.ndarray, qubo_terms: List[Tuple[int, int, float]]) -> float:
    e = 0.0
    for i, j, q in qubo_terms:
        e += q * x[i] * x[j]
    return e


def ising_energy(s: np.ndarray, h: np.ndarray, J: np.ndarray, const: float = 0.0) -> float:
    
    return const + float(h @ s) + 0.5 * float(s @ (J @ s))


def spins_from_binary(x: np.ndarray) -> np.ndarray:
    # x ∈ {0,1} → s ∈ {-1,1}: s = 2x - 1
    return 2 * x - 1


# sanity-check: энергия QUBO и Ising для пары простых конфигураций
x_zero = np.zeros(num_variables, dtype=int)
s_zero = spins_from_binary(x_zero)

x_one = np.ones(num_variables, dtype=int)
s_one = spins_from_binary(x_one)

{
    'E_qubo_zero': qubo_energy(x_zero, qubo_Q),
    'E_ising_zero': ising_energy(s_zero, h_vec, J_mat, qubo_const),
    'E_qubo_one': qubo_energy(x_one, qubo_Q),
    'E_ising_one': ising_energy(s_one, h_vec, J_mat, qubo_const),
}


{'E_qubo_zero': np.float64(0.0),
 'E_ising_zero': 0.0,
 'E_qubo_one': np.float64(42437543.86),
 'E_ising_one': 42437543.86}

In [84]:
qubo_dense_matrix = [[0.0 for _ in range(num_variables)] for _ in range(num_variables)]

for i, j, q in qubo_Q:
    qubo_dense_matrix[i][j] += float(q)
    if i != j:
        qubo_dense_matrix[j][i] += float(q)

qubo_dense_path = QUBO_DIR / 'universal_qubo_dense_matrix.json'
with qubo_dense_path.open('w', encoding='utf-8') as f:
    json.dump(qubo_dense_matrix, f, ensure_ascii=False, indent=2)

{
    'dense_matrix_path': str(qubo_dense_path),
    'shape': (len(qubo_dense_matrix), len(qubo_dense_matrix[0]) if qubo_dense_matrix else 0),
}

pd.DataFrame(qubo_dense_matrix)



,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,-23940.24,400000.00,400000.00,400000.00,400000.00,400000.00,400000.00,400000.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
1,400000.00,-26463.43,400000.00,400000.00,400000.00,400000.00,400000.00,400000.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2,400000.00,400000.00,-24030.24,400000.00,400000.00,400000.00,400000.00,400000.00,0.00,0.00,...,80000.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
3,400000.00,400000.00,400000.00,-23999.04,400000.00,400000.00,400000.00,400000.00,0.00,0.00,...,0.00,80000.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
4,400000.00,400000.00,400000.00,400000.00,-23940.24,400000.00,400000.00,400000.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
5,400000.00,400000.00,400000.00,400000.00,400000.00,-26463.43,400000.00,400000.00,400000.00,400000.00,...,0.00,0.00,0.00,80000.00,0.00,0.00,0.00,0.00,0.00,0.00
6,400000.00,400000.00,400000.00,400000.00,400000.00,400000.00,-24030.24,400000.00,0.00,0.00,...,80000.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
7,400000.00,400000.00,400000.00,400000.00,400000.00,400000.00,400000.00,-23999.04,0.00,0.00,...,0.00,80000.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
8,0.00,0.00,0.00,0.00,0.00,400000.00,0.00,0.00,-40897.91,400000.00,...,0.00,0.00,80000.00,0.00,0.00,0.00,80000.00,0.00,0.00,0.00
9,0.00,0.00,0.00,0.00,0.00,400000.00,0.00,0.00,400000.00,-41442.73,...,0.00,0.00,0.00,80000.00,0.00,0.00,0.00,80000.00,0.00,0.00


## D-Wave SA


In [85]:
import dimod
import neal

Q_dict = {}
for i, j, value in qubo_Q:
    # аккумулируем на случай повторов одной пары
    Q_dict[(i, j)] = Q_dict.get((i, j), 0.0) + float(value)

bqm = dimod.BinaryQuadraticModel.from_qubo(Q_dict)
sampler = neal.SimulatedAnnealingSampler()

num_reads = 100
num_sweeps = 2000

sampleset = sampler.sample(
    bqm,
    num_reads=num_reads,
    num_sweeps=num_sweeps,
    beta_schedule_type="geometric",   # можно поиграть позже
)

best = sampleset.first
best_sample = best.sample   # dict: var_index -> 0/1
best_energy = best.energy

best_energy, list(best_sample.items())[:10]




(np.float64(-321578.0300000012),
 [(0, np.int8(0)),
  (1, np.int8(0)),
  (2, np.int8(0)),
  (3, np.int8(0)),
  (4, np.int8(0)),
  (5, np.int8(0)),
  (6, np.int8(0)),
  (7, np.int8(1)),
  (8, np.int8(0)),
  (9, np.int8(0))])

In [86]:
def decode_sample_to_schedule(sample, var_index_to_candidate_idx, candidate_df):
    selected_cand_idx = [
        var_index_to_candidate_idx[v]
        for v, bit in sample.items()
        if int(bit) == 1
    ]
    schedule = candidate_df.loc[selected_cand_idx].copy()
    return schedule
sa_schedule = decode_sample_to_schedule(
    best_sample,
    qubo_var_index_map_int,
    universal_candidate_scored_df,
)

print(sa_schedule["pred_revenue"].sum())
sa_schedule


321578.02999999997


,show_id,slot_id,cinema_id,hall_id,hall_capacity,movie_id,genre,age_rating,runtime_min,release_date,...,month,holiday_flag,release_age_days,lead_time_days,prime_time_flag,runtime_bucket,screening_number_for_movie,pred_occupancy_rate,pred_sold_tickets,pred_revenue
7,hall_small__s2__movie_horror_01,hall_small__s2,cinema_moscow_toy,hall_small,90,movie_horror_01,horror,18+,98,2026-03-05,...,1,False,-49,7.0,False,medium,2,0.531352,48,23999.04
10,hall_small__s3__movie_family_01,hall_small__s3,cinema_moscow_toy,hall_small,90,movie_family_01,animation,6+,92,2026-02-20,...,1,False,-36,7.0,True,medium,3,0.913910,82,41051.66
15,hall_small__s4__movie_horror_01,hall_small__s4,cinema_moscow_toy,hall_small,90,movie_horror_01,horror,18+,98,2026-03-05,...,1,False,-49,7.0,True,medium,4,0.912608,82,40998.36
17,hall_large__s1__movie_dune_2,hall_large__s1,cinema_moscow_toy,hall_large,180,movie_dune_2,sci_fi,16+,166,2026-03-01,...,1,False,-45,7.0,False,long,1,0.570182,103,51428.93
27,hall_large__s3__movie_horror_01,hall_large__s3,cinema_moscow_toy,hall_large,180,movie_horror_01,horror,18+,98,2026-03-05,...,1,False,-49,7.0,True,medium,3,0.908486,164,81996.72
30,hall_large__s4__movie_family_01,hall_large__s4,cinema_moscow_toy,hall_large,180,movie_family_01,animation,6+,92,2026-02-20,...,1,False,-36,7.0,True,medium,4,0.908999,164,82103.32


In [99]:
def intervals_overlap(start_a, end_a, start_b, end_b):
    return max(start_a, start_b) < min(end_a, end_b)


def find_physical_hall_conflicts(schedule_df):
    work_df = schedule_df.copy().sort_values(
        ["cinema_id", "hall_id", "show_datetime"]
    )

    conflicts = []

    for (cinema_id, hall_id), group in work_df.groupby(["cinema_id", "hall_id"]):
        rows = list(group.itertuples())

        for i in range(len(rows)):
            a = rows[i]
            for j in range(i + 1, len(rows)):
                b = rows[j]

                # так как отсортировано по start time, можно рано выйти
                if a.end_datetime <= b.show_datetime:
                    break

                if intervals_overlap(
                    a.show_datetime,
                    a.end_datetime,
                    b.show_datetime,
                    b.end_datetime,
                ):
                    conflicts.append(
                        {
                            "cinema_id": cinema_id,
                            "hall_id": hall_id,
                            "candidate_idx_a": a.Index,
                            "candidate_idx_b": b.Index,
                            "movie_id_a": a.movie_id,
                            "movie_id_b": b.movie_id,
                            "start_a": a.show_datetime,
                            "end_a": a.end_datetime,
                            "start_b": b.show_datetime,
                            "end_b": b.end_datetime,
                            "pred_revenue_a": a.pred_revenue,
                            "pred_revenue_b": b.pred_revenue,
                        }
                    )

    return pd.DataFrame(conflicts)
def find_same_movie_cross_hall_conflicts(schedule_df):
    work_df = schedule_df.copy().sort_values(
        ["cinema_id", "movie_id", "show_datetime"]
    )

    conflicts = []

    for (cinema_id, movie_id), group in work_df.groupby(["cinema_id", "movie_id"]):
        rows = list(group.itertuples())

        for i in range(len(rows)):
            a = rows[i]
            for j in range(i + 1, len(rows)):
                b = rows[j]

                # так как отсортировано по start time, можно рано выйти
                if a.end_datetime <= b.show_datetime:
                    break

                # интересуют только разные залы
                if a.hall_id == b.hall_id:
                    continue

                if intervals_overlap(
                    a.show_datetime,
                    a.end_datetime,
                    b.show_datetime,
                    b.end_datetime,
                ):
                    conflicts.append(
                        {
                            "cinema_id": cinema_id,
                            "movie_id": movie_id,
                            "candidate_idx_a": a.Index,
                            "candidate_idx_b": b.Index,
                            "hall_id_a": a.hall_id,
                            "hall_id_b": b.hall_id,
                            "start_a": a.show_datetime,
                            "end_a": a.end_datetime,
                            "start_b": b.show_datetime,
                            "end_b": b.end_datetime,
                            "pred_revenue_a": a.pred_revenue,
                            "pred_revenue_b": b.pred_revenue,
                        }
                    )

    return pd.DataFrame(conflicts)

hall_conflicts_sa = find_physical_hall_conflicts(sa_schedule)
print("D-Wave SA")
print("SA revenue:", sa_schedule["pred_revenue"].sum())
print("SA shows:", len(sa_schedule))
print("SA hall conflicts:", len(hall_conflicts_sa))

D-Wave SA
SA revenue: 321578.02999999997
SA shows: 6
SA hall conflicts: 0


## Openjij Sa

In [101]:
import openjij as oj
import numpy as np
import pandas as pd


def build_openjij_ising_inputs(
    h_vec: np.ndarray,
    J_mat: np.ndarray,
) -> tuple[dict[int, float], dict[tuple[int, int], float]]:
    num_vars = len(h_vec)
    h = {i: float(h_vec[i]) for i in range(num_vars)}
    J = {}

    for i in range(num_vars):
        for j in range(i + 1, num_vars):
            if float(J_mat[i, j]) != 0.0:
                J[(i, j)] = float(J_mat[i, j])

    return h, J


def spins_to_binary_sample(state):
    state = np.asarray(state, dtype=int)
    x = (state + 1) // 2
    return {i: int(x[i]) for i in range(len(x))}


def decode_sample_to_schedule(sample, var_index_to_candidate_idx, candidate_df):
    selected_cand_idx = [
        var_index_to_candidate_idx[v]
        for v, bit in sample.items()
        if int(bit) == 1
    ]
    schedule = candidate_df.loc[selected_cand_idx].copy()
    return schedule


def intervals_overlap(start_a, end_a, start_b, end_b):
    return max(start_a, start_b) < min(end_a, end_b)


def find_physical_hall_conflicts(schedule_df):
    work_df = schedule_df.copy().sort_values(
        ["cinema_id", "hall_id", "show_datetime"]
    )

    conflicts = []

    for (cinema_id, hall_id), group in work_df.groupby(["cinema_id", "hall_id"]):
        rows = list(group.itertuples())

        for i in range(len(rows)):
            a = rows[i]
            for j in range(i + 1, len(rows)):
                b = rows[j]

                if a.end_datetime <= b.show_datetime:
                    break

                if intervals_overlap(
                    a.show_datetime,
                    a.end_datetime,
                    b.show_datetime,
                    b.end_datetime,
                ):
                    conflicts.append(
                        {
                            "cinema_id": cinema_id,
                            "hall_id": hall_id,
                            "candidate_idx_a": a.Index,
                            "candidate_idx_b": b.Index,
                            "movie_id_a": a.movie_id,
                            "movie_id_b": b.movie_id,
                            "start_a": a.show_datetime,
                            "end_a": a.end_datetime,
                            "start_b": b.show_datetime,
                            "end_b": b.end_datetime,
                            "pred_revenue_a": a.pred_revenue,
                            "pred_revenue_b": b.pred_revenue,
                        }
                    )

    return pd.DataFrame(conflicts)


def find_same_movie_cross_hall_conflicts(schedule_df):
    work_df = schedule_df.copy().sort_values(
        ["cinema_id", "movie_id", "show_datetime"]
    )

    conflicts = []

    for (cinema_id, movie_id), group in work_df.groupby(["cinema_id", "movie_id"]):
        rows = list(group.itertuples())

        for i in range(len(rows)):
            a = rows[i]
            for j in range(i + 1, len(rows)):
                b = rows[j]

                if a.end_datetime <= b.show_datetime:
                    break

                if a.hall_id == b.hall_id:
                    continue

                if intervals_overlap(
                    a.show_datetime,
                    a.end_datetime,
                    b.show_datetime,
                    b.end_datetime,
                ):
                    conflicts.append(
                        {
                            "cinema_id": cinema_id,
                            "movie_id": movie_id,
                            "candidate_idx_a": a.Index,
                            "candidate_idx_b": b.Index,
                            "hall_id_a": a.hall_id,
                            "hall_id_b": b.hall_id,
                            "start_a": a.show_datetime,
                            "end_a": a.end_datetime,
                            "start_b": b.show_datetime,
                            "end_b": b.end_datetime,
                            "pred_revenue_a": a.pred_revenue,
                            "pred_revenue_b": b.pred_revenue,
                        }
                    )

    return pd.DataFrame(conflicts)


def prepare_schedule_view(schedule_df):
    cols = [
        c for c in [
            "cinema_id",
            "hall_id",
            "movie_id",
            "show_datetime",
            "end_datetime",
            "pred_revenue",
            "format",
            "hall_capacity",
            "slot_id",
        ]
        if c in schedule_df.columns
    ]

    return (
        schedule_df[cols]
        .sort_values(["cinema_id", "hall_id", "show_datetime"])
        .reset_index()
        .rename(columns={"index": "candidate_idx"})
    )


def evaluate_schedule(name, spins, openjij_energy):
    spins = np.asarray(spins, dtype=int)

    assert len(spins) == num_variables, (
        f"Spin vector length {len(spins)} != num_variables {num_variables}"
    )

    sample = spins_to_binary_sample(spins)
    schedule = decode_sample_to_schedule(
        sample,
        qubo_var_index_map_int,
        universal_candidate_scored_df,
    )

    reconstructed_ising_energy = ising_energy(
        spins,
        h_vec,
        J_mat,
        qubo_const,
    )

    hall_conflicts_df = find_physical_hall_conflicts(schedule)
    same_movie_conflicts_df = find_same_movie_cross_hall_conflicts(schedule)

    metrics = {
        "solver": name,
        "openjij_energy": float(openjij_energy),
        "reconstructed_ising_energy": float(reconstructed_ising_energy),
        "shows": int(len(schedule)),
        "revenue": float(schedule["pred_revenue"].sum()),
        "hall_conflicts": int(len(hall_conflicts_df)),
        "same_movie_conflicts": int(len(same_movie_conflicts_df)),
    }

    return schedule, hall_conflicts_df, same_movie_conflicts_df, metrics


def print_solution_summary(metrics):
    print("=" * 80)
    print(metrics["solver"])
    print("=" * 80)
    print("OpenJij energy:", metrics["openjij_energy"])
    print("Reconstructed Ising energy:", metrics["reconstructed_ising_energy"])
    print("Revenue:", metrics["revenue"])
    print("Shows:", metrics["shows"])
    print("Hall conflicts:", metrics["hall_conflicts"])
    print("Same-movie cross-hall conflicts:", metrics["same_movie_conflicts"])

    if metrics["hall_conflicts"] == 0 and metrics["same_movie_conflicts"] == 0:
        print("Status: feasible schedule without detected conflicts")
    elif metrics["hall_conflicts"] > 0:
        print("Status: infeasible because physical hall overlaps are present")
    else:
        print("Status: physically feasible, but same-movie cross-hall overlaps exist")


# -----------------------------
# Build OpenJij inputs
# -----------------------------
h_oj, J_oj = build_openjij_ising_inputs(h_vec, J_mat)

print("num_variables:", num_variables)
print("len(h_oj):", len(h_oj))
print("num_J_terms:", len(J_oj))

num_reads = 100
num_sweeps = 3000

# -----------------------------
# OpenJij SA
# -----------------------------
sa_sampler_oj = oj.SASampler()
oj_sa_response = sa_sampler_oj.sample_ising(
    h_oj,
    J_oj,
    num_reads=num_reads,
    num_sweeps=num_sweeps,
)

oj_sa_states = np.asarray(oj_sa_response.states)
oj_sa_energies = np.asarray(oj_sa_response.energies, dtype=float)

sa_best_idx = int(np.argmin(oj_sa_energies))
sa_best_state = np.asarray(oj_sa_states[sa_best_idx], dtype=int)
sa_best_energy = float(oj_sa_energies[sa_best_idx])

sa_schedule_oj, sa_hall_conflicts_oj, sa_same_movie_conflicts_oj, sa_metrics = evaluate_schedule(
    "OpenJij SA",
    sa_best_state,
    sa_best_energy,
)

sa_schedule_view = prepare_schedule_view(sa_schedule_oj)

# -----------------------------
# OpenJij SQA
# -----------------------------
sqa_sampler_oj = oj.SQASampler()
oj_sqa_response = sqa_sampler_oj.sample_ising(
    h_oj,
    J_oj,
    num_reads=num_reads,
    num_sweeps=num_sweeps,
)

oj_sqa_states = np.asarray(oj_sqa_response.states)
oj_sqa_energies = np.asarray(oj_sqa_response.energies, dtype=float)

sqa_best_idx = int(np.argmin(oj_sqa_energies))
sqa_best_state = np.asarray(oj_sqa_states[sqa_best_idx], dtype=int)
sqa_best_energy = float(oj_sqa_energies[sqa_best_idx])

sqa_schedule_oj, sqa_hall_conflicts_oj, sqa_same_movie_conflicts_oj, sqa_metrics = evaluate_schedule(
    "OpenJij SQA",
    sqa_best_state,
    sqa_best_energy,
)

sqa_schedule_view = prepare_schedule_view(sqa_schedule_oj)

# -----------------------------
# Final comparison table
# -----------------------------
comparison_df = (
    pd.DataFrame([sa_metrics, sqa_metrics])
    .sort_values(
        ["hall_conflicts", "same_movie_conflicts", "revenue"],
        ascending=[True, True, False],
    )
    .reset_index(drop=True)
)

print("\nFINAL METRICS")
display(comparison_df)

# -----------------------------
# Summaries
# -----------------------------
print()
print_solution_summary(sa_metrics)
print()
print_solution_summary(sqa_metrics)

# -----------------------------
# Final schedules
# -----------------------------
print("\nOPENJIJ SA SCHEDULE")
display(sa_schedule_view)

print("\nOPENJIJ SQA SCHEDULE")
display(sqa_schedule_view)

# -----------------------------
# Conflict analysis
# -----------------------------
print("\nOPENJIJ SA HALL CONFLICTS")
display(sa_hall_conflicts_oj)

print("\nOPENJIJ SA SAME-MOVIE CROSS-HALL CONFLICTS")
display(sa_same_movie_conflicts_oj)

print("\nOPENJIJ SQA HALL CONFLICTS")
display(sqa_hall_conflicts_oj)

print("\nOPENJIJ SQA SAME-MOVIE CROSS-HALL CONFLICTS")
display(sqa_same_movie_conflicts_oj)


num_variables: 32
len(h_oj): 32
num_J_terms: 134

FINAL METRICS


,solver,openjij_energy,reconstructed_ising_energy,shows,revenue,hall_conflicts,same_movie_conflicts
0,OpenJij SA,-10540847.63,-322075.70,6,322075.70,0,0
1,OpenJij SQA,-10540280.71,-321508.78,6,321508.78,0,0



OpenJij SA
OpenJij energy: -10540847.630000003
Reconstructed Ising energy: -322075.69999999925
Revenue: 322075.7
Shows: 6
Hall conflicts: 0
Same-movie cross-hall conflicts: 0
Status: feasible schedule without detected conflicts

OpenJij SQA
OpenJij energy: -10540280.71
Reconstructed Ising energy: -321508.7800000012
Revenue: 321508.77999999997
Shows: 6
Hall conflicts: 0
Same-movie cross-hall conflicts: 0
Status: feasible schedule without detected conflicts

OPENJIJ SA SCHEDULE


,candidate_idx,cinema_id,hall_id,movie_id,show_datetime,end_datetime,pred_revenue,format,hall_capacity,slot_id
0,17,cinema_moscow_toy,hall_large,movie_dune_2,2026-01-15 13:30:00,2026-01-15 16:16:00,51428.93,IMAX,180,hall_large__s1
1,26,cinema_moscow_toy,hall_large,movie_family_01,2026-01-15 18:00:00,2026-01-15 19:32:00,82103.32,IMAX,180,hall_large__s3
2,30,cinema_moscow_toy,hall_large,movie_family_01,2026-01-15 20:00:00,2026-01-15 21:32:00,82103.32,IMAX,180,hall_large__s4
3,7,cinema_moscow_toy,hall_small,movie_horror_01,2026-01-15 15:00:00,2026-01-15 16:38:00,23999.04,2D,90,hall_small__s2
4,11,cinema_moscow_toy,hall_small,movie_horror_01,2026-01-15 18:00:00,2026-01-15 19:38:00,40998.36,2D,90,hall_small__s3
5,13,cinema_moscow_toy,hall_small,movie_dune_2,2026-01-15 20:00:00,2026-01-15 22:46:00,41442.73,2D,90,hall_small__s4



OPENJIJ SQA SCHEDULE


...

""



OPENJIJ SQA HALL CONFLICTS


""



OPENJIJ SQA SAME-MOVIE CROSS-HALL CONFLICTS


""
